In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os
from corner import corner

# single run plots

In [ ]:
#rundir  = 'wdmruns/noise-conf/'
#rundir  = 'wdmruns-coarse/stat-noise-conf-pl/'
#rundir  = 'wdmruns-coarse/noise-conf-pl-13/'
rundir  = 'wdmruns-coarse/noise-conf-pl-169/'
rundir = 'wdmruns-coarse2/stat-noise-conf-ln-detection/'
rundir = 'wdmruns-coarse2/noise-conf-ln-169-detection/'
rundir = 'wdmruns-coarse3/stat-noise-conf-ln-detection/'
rundir = 'wdmruns-coarse3/noise-conf-ln-ws169-detection/'
rundir = 'wdmruns-coarse3/noise-conf-ln-ws169-avg-detection/'
#rundir = 'wdmruns-coarse4/noise-conf-ln-1-detection-long/'
#rundir = 'wdmruns-coarse3/statws-noise-conf-ln-detection/'

## check out the data

In [ ]:
datadir = os.path.join(rundir,'data')
os.listdir(datadir)

In [ ]:
scaleogram_data = np.loadtxt(os.path.join(datadir,'scaleogram_data.dat'))
full_noise      = np.loadtxt(os.path.join(datadir,'full_noise_model.dat'))
dwt_data        = np.loadtxt(os.path.join(datadir,'dwt_data.dat'))

In [ ]:
# cols are t, f, |X|^2, |Y|^2, |Z|^2

In [ ]:
full_noise.shape

In [ ]:
dwt_data.shape

In [ ]:
dwt_data[-1,0]

In [ ]:
import matplotlib_inline.backend_inline
matplotlib_inline.backend_inline.set_matplotlib_formats('svg')
from matplotlib import colors
%matplotlib inline

def get_Nt_Nf(datafile_arr):
    # cols are t, f, ...
    Nf = np.unique(datafile_arr[:,1]).shape[0]
    Nt = np.unique(datafile_arr[:,0]).shape[0]
    assert Nt*Nf == datafile_arr.shape[0]
    return Nt,Nf

def wavelet_specgram(t, f, pz, Nt = 512, clip = 0, lognorm=False, cmap='bwr', cx_norm='abs', logf=False):
    Nf = t.shape[0] // Nt
    if t.shape[0] != Nt*Nf or f.shape[0] != Nt*Nf:
        print("Nt or Nf is wrong")
    z = pz.copy()
    if lognorm:
        z = np.ma.masked_less_equal(z.astype(float), 0)
        lo, hi = np.nanpercentile(z.compressed(), [clip, 100-clip])
        norm = colors.AsinhNorm(vmin=lo, vmax = hi, clip=True)
    else:
        lo, hi = np.nanpercentile(z, [clip, 100-clip])
        norm = colors.Normalize(vmin=lo, vmax = hi, clip=True)
    #plt.imshow(z.reshape((Nf,Nt)),cmap=cmap,norm=norm,aspect=9/16*Nt/Nf,interpolation='none')
    fig, ax = plt.subplots()
    shading = 'auto'
    tp = t[::Nf]
    fp = f[:Nf]
    if Nt == 1:
        tp = np.array([tp[0], dwt_data[-1,0]])
        fp = np.hstack([fp, fp[-1] + (fp[1]-fp[0])])
    m = ax.pcolormesh(tp, fp, z.reshape((Nt,Nf)).T, cmap=cmap, norm=norm, shading=shading, snap=True, rasterized=True)
    if logf:
        ax.set_yscale('log')
    fig.colorbar(m,ax=ax)
    plt.xlabel('t')
    plt.ylabel('f')
    plt.show()
    

In [ ]:
Nt,Nf = get_Nt_Nf(full_noise)

In [ ]:
full_noise[:Nf,1]

In [ ]:
wavelet_specgram(full_noise[:,0], full_noise[:,1], full_noise[:,2], Nt=Nt, lognorm=True, clip = 5, cmap='viridis', logf=False)

In [ ]:
wavelet_specgram(scaleogram_data[:,0],
                 scaleogram_data[:,1],
                 scaleogram_data[:,2],
                 Nt=Nt,
                 lognorm=False,
                cmap = 'Blues',
                clip = 5)

In [ ]:
# gaussianity test of generated data
# fudge_factor = 3 # implemented in code
counts,bins,_ = plt.hist(dwt_data[:,4] / np.sqrt(full_noise[:,4]), density=True, bins=100)
x = np.linspace(bins[0],bins[-1],100)
plt.plot(x, np.exp(-x**2 / 2)*np.max(counts), color='k', linestyle = '--')
plt.yscale('log')

In [ ]:
  whitened = dwt_data[:,4] / np.sqrt(full_noise[:,4])
  print(f"Whitened variance: {np.var(whitened):.4f}")

In [ ]:
qeff_fname = os.path.join(datadir, "coarse_Qeff.dat")
if os.path.exists(qeff_fname):
    qeff = np.loadtxt(qeff_fname)
    Nt,Nf = get_Nt_Nf(qeff)
    wavelet_specgram(qeff[:,0],
                     qeff[:,1],
                     qeff[:,2],
                     Nt=Nt,
                     lognorm=False,
                    cmap = 'Blues',
                    clip = 0)

## chains

In [ ]:
chaindir = os.path.join(rundir, 'chains')

In [ ]:
os.listdir(chaindir)

In [ ]:
sgwb = None
if os.path.exists(os.path.join(chaindir, 'sgwb_chain.dat')):
    sgwb = np.loadtxt(os.path.join(chaindir, 'sgwb_chain.dat'))
galx = None
if os.path.exists(os.path.join(chaindir, 'foreground_chain.dat')):
    galx = np.loadtxt(os.path.join(chaindir, 'foreground_chain.dat'))
inst = np.loadtxt(os.path.join(chaindir, 'noise_chain.dat'))

In [ ]:
sgwb.shape

In [ ]:
inst.shape

In [ ]:
galx.shape

In [ ]:
plt.plot(inst[:,1])

In [ ]:
# sgwb trace
nparams = sgwb.shape[1]-2
fig, ax = plt.subplots(nparams,1)
for i in range(nparams):
    ax[i].scatter(range(sgwb.shape[0]), sgwb[0:,i+2], s=0.1)

In [ ]:
# inst trace
burn = 0
fig, ax = plt.subplots(12,1)
for i in range(12):
    ax[i].scatter(range(inst.shape[0]-burn), inst[burn:,i+2], s=0.1)

In [ ]:
burn = 0
fig, ax = plt.subplots(5,1)
for i in range(5):
    ax[i].scatter(range(galx.shape[0]-burn), galx[burn:,i+2], s=0.1)

In [ ]:
from corner import corner

In [ ]:
burn = 0
sgwb_chain = np.array(
    [
        sgwb[burn:,2], # logA
        sgwb[burn:,3], # alpha
    ]).T
sgwb_labels = [r'$\log A_p$', r'$\alpha$']

In [ ]:
inst_chain = np.array(
    [
        inst[burn:,i+2]
        for i in range(12)
    ]).T
inst_labels = [r'$\log S_{\mathrm{acc},12}$',
r'$\log S_{\mathrm{acc},21}$',
r'$\log S_{\mathrm{acc},13}$',
r'$\log S_{\mathrm{acc},31}$',
r'$\log S_{\mathrm{acc},23}$', 
r'$\log S_{\mathrm{acc},32}$', 
r'$\log S_{\mathrm{oms},12}$', 
r'$\log S_{\mathrm{oms},21}$', 
r'$\log S_{\mathrm{oms},13}$', 
r'$\log S_{\mathrm{oms},31}$', 
r'$\log S_{\mathrm{oms},23}$', 
r'$\log S_{\mathrm{oms},32}$']

In [ ]:
galx_chain = np.array(
    [
        galx[burn:,i+2]
        for i in range(5)
    ]).T
galx_labels = [r'$\log A$', r'$f_1$', r'$\alpha$', r'$f_k$', r'$f_2$']

In [ ]:
truths = sgwb[0,2:]
corner(sgwb_chain, labels = sgwb_labels, truths=truths, truth_color = 'red')
plt.show()

In [ ]:
truths = inst[0,2:]
corner(inst_chain, labels = inst_labels, truths = truths, truth_color='red')
plt.show()

In [ ]:
truths = galx[0,2:]
corner(galx_chain, labels = galx_labels, truths = truths, truth_color = 'red')
plt.show()

# compare coarse runs - powerlaw

In [ ]:
rundirs  = [#'wdmruns-coarse/stat-noise-conf-pl-detection/',
            #'wdmruns-coarse/noise-conf-pl-13/',
            #'wdmruns-coarse/noise-conf-pl-169-detection/',
#            'wdmruns-coarse6/stat-noise-conf-pl/',
            'wdmruns-coarse6/noise-conf-pl-169/',
           ]
chaindirs = [os.path.join(rundir, 'chains') for rundir in rundirs]
labels = [
    #'stationary',
    #'Ncoarse=312',
    'non-stationary',
         ]
colors = [
    'black',
          #'red',
          'blue',
         ]

In [ ]:
inst_labels = [r'$\log S_{\mathrm{acc},12}$',
r'$\log S_{\mathrm{acc},21}$',
r'$\log S_{\mathrm{acc},13}$',
r'$\log S_{\mathrm{acc},31}$',
r'$\log S_{\mathrm{acc},23}$', 
r'$\log S_{\mathrm{acc},32}$', 
r'$\log S_{\mathrm{oms},12}$', 
r'$\log S_{\mathrm{oms},21}$', 
r'$\log S_{\mathrm{oms},13}$', 
r'$\log S_{\mathrm{oms},31}$', 
r'$\log S_{\mathrm{oms},23}$', 
r'$\log S_{\mathrm{oms},32}$']

In [ ]:
galx_labels = [r'$\log A$', r'$f_1$', r'$\alpha$', r'$f_k$', r'$f_2$']
sgwb_labels = [r'$\log A_p$', r'$\alpha$']

In [ ]:
from chainconsumer import Chain, ChainConfig, PlotConfig, Truth
import chainconsumer
import pandas as pd

In [ ]:
c = chainconsumer.ChainConsumer()
for chaindir,label in zip(chaindirs, labels):
    inst = np.loadtxt(os.path.join(chaindir, 'noise_chain.dat'))
    
    inst_oms_chain = np.array(
    [
        inst[burn:,i+2+6]
        for i in range(6)
    ]).T
    inst_oms_df = pd.DataFrame(inst_oms_chain, columns = inst_labels[6:])
    truths = inst[0,8:]
    c.add_chain(Chain(samples=inst_oms_df, name=label))
c.add_truth(Truth(location = {inst_labels[i+6] : truths[i] for i in range(6)}))
c.plotter.set_config(PlotConfig(contour_label_font_size=5, dpi=100, legend_kwargs={'fontsize':20}, legend_location=(1,3), plot_hists=False))
c.plotter.plot()
plt.savefig("oms_contour.png")

In [ ]:
c = chainconsumer.ChainConsumer()
for chaindir,label in zip(chaindirs, labels):
    inst = np.loadtxt(os.path.join(chaindir, 'noise_chain.dat'))
    
    inst_tm_chain = np.array(
    [
        inst[burn:,i+2]
        for i in range(6)
    ]).T
    inst_tm_df = pd.DataFrame(inst_tm_chain, columns = inst_labels[:6])
    truths = inst[0,2:8]
    c.add_chain(Chain(samples=inst_tm_df, name=label))
c.add_truth(Truth(location = {inst_labels[i] : truths[i] for i in range(6)}))
c.plotter.set_config(PlotConfig(contour_label_font_size=5, dpi=100, legend_kwargs={'fontsize':20}, legend_location=(1,3), plot_hists=False))
c.plotter.plot()
plt.savefig("tm_contour.png")

In [ ]:
inst = np.loadtxt(os.path.join(chaindirs[0], 'noise_chain.dat'))
inst_chain = np.array(
[
    inst[burn:,i+2]
    for i in range(12)
]).T
truths = inst[0,2:]
old_corner = corner(inst_chain, labels = inst_labels, truths = truths, truth_color='red', label=labels[0], color=colors[0], hist_kwargs={'density':True})

for (chdir, label, color) in zip(chaindirs[1:], labels[1:], colors[1:]):
    inst = np.loadtxt(os.path.join(chdir, 'noise_chain.dat'))
    inst_chain = np.array(
    [
        inst[burn:,i+2]
        for i in range(12)
    ]).T
    truths = inst[0,2:]
    corner(inst_chain, fig= old_corner, labels = inst_labels, color=color, label=label, hist_kwargs={'density':True})
    

In [ ]:
galx = np.loadtxt(os.path.join(chaindirs[0], 'foreground_chain.dat'))
galx_chain = np.array(
[
    galx[burn:,i+2]
    for i in range(5)
]).T
truths = galx[0,2:]
old_corner = corner(galx_chain, labels = galx_labels, truths = truths, truth_color='red', label=labels[0], color=colors[0], hist_kwargs={'density':True})

for (chdir, label, color) in zip(chaindirs[1:], labels[1:], colors[1:]):
    galx = np.loadtxt(os.path.join(chdir, 'foreground_chain.dat'))
    galx_chain = np.array(
    [
        galx[burn:,i+2]
        for i in range(5)
    ]).T
    corner(galx_chain, fig= old_corner, labels = galx_labels, color=color, label=label, hist_kwargs={'density':True})
    

In [ ]:
c = chainconsumer.ChainConsumer()
for chaindir,label in zip(chaindirs, labels):
    galx = np.loadtxt(os.path.join(chaindir, 'foreground_chain.dat'))
    
    galx_chain = np.array(
    [
        galx[burn:,i+2]
        for i in range(5)
    ]).T
    galx_df = pd.DataFrame(galx_chain, columns = galx_labels)
    truths = galx[0,2:]
    c.add_chain(Chain(samples=galx_df, name=label))
c.add_truth(Truth(location = {galx_labels[i] : truths[i] for i in range(5)}))
c.plotter.set_config(PlotConfig(contour_label_font_size=16, dpi=100, legend_kwargs={'fontsize':20}, legend_location=(1,3), plot_hists=False))
c.plotter.plot()
plt.savefig("galx_contour.png")

In [ ]:
sgwb = np.loadtxt(os.path.join(chaindirs[0], 'sgwb_chain.dat'))
sgwb_chain = np.array(
[
    sgwb[burn:,i+2]
    for i in range(2)
]).T
truths = sgwb[0,2:]
old_corner = corner(sgwb_chain, labels = sgwb_labels, truths = truths, truth_color='red', label=labels[0], color=colors[0], hist_kwargs={'density':True})

for (chdir, label, color) in zip(chaindirs[1:], labels[1:], colors[1:]):
    sgwb = np.loadtxt(os.path.join(chdir, 'sgwb_chain.dat'))
    sgwb_chain = np.array(
    [
        sgwb[burn:,i+2]
        for i in range(2)
    ]).T
    corner(sgwb_chain, fig= old_corner, labels = sgwb_labels, color=color, label=label, hist_kwargs={'density':True})
    

In [ ]:
sgwb[0,:]

In [ ]:
c = chainconsumer.ChainConsumer()
for chaindir,label in zip(chaindirs, labels):
    sgwb = np.loadtxt(os.path.join(chaindir, 'sgwb_chain.dat'))
    
    sgwb_chain = np.array(
    [
        sgwb[burn:,i+2]
        for i in range(2)
    ]).T
    sgwb_df = pd.DataFrame(sgwb_chain, columns = sgwb_labels)
    truths = [-20.0,2/3.]
    c.add_chain(Chain(samples=sgwb_df, name=label))
c.add_truth(Truth(location = {sgwb_labels[i] : truths[i] for i in range(2)}))
c.plotter.set_config(
    PlotConfig(
        contour_label_font_size=16,
        dpi=100,
        legend_kwargs={'fontsize':20, 'loc': 'lower right'},
        plot_hists=False,
    )
)
c.plotter.plot()
plt.savefig("powerlaw_contour.png")

In [ ]:
import matplotlib_inline.backend_inline
matplotlib_inline.backend_inline.set_matplotlib_formats('svg')
from matplotlib import colors
%matplotlib inline

def get_Nt_Nf(datafile_arr):
    # cols are t, f, ...
    Nf = np.unique(datafile_arr[:,1]).shape[0]
    Nt = np.unique(datafile_arr[:,0]).shape[0]
    assert Nt*Nf == datafile_arr.shape[0]
    return Nt,Nf

# TODO: separate real/imag FFT coeffs in each layer
def wavelet_specgram(t, f, pz, Nt = 512, clip = 0, lognorm=False, logf=False, cmap='bwr', cx_norm='abs', title=None, tunits='s', savefname=None):
    Nf = t.shape[0] // Nt
    if t.shape[0] != Nt*Nf or f.shape[0] != Nt*Nf:
        print("Nt or Nf is wrong")
    z = pz.copy()
    if lognorm:
        z = np.ma.masked_less_equal(z.astype(float), 0)
        lo, hi = np.nanpercentile(z.compressed(), [clip, 100-clip])
        norm = colors.AsinhNorm(vmin=lo, vmax = hi, clip=True)
    else:
        lo, hi = np.nanpercentile(z, [clip, 100-clip])
        norm = colors.Normalize(vmin=lo, vmax = hi, clip=True)
    #plt.imshow(z.reshape((Nf,Nt)),cmap=cmap,norm=norm,aspect=9/16*Nt/Nf,interpolation='none')
    fig, ax = plt.subplots()
    if tunits == 's':
        tp = t[::Nf]
        xlabel = 't [s]'
    elif tunits == 'days':
        tp = t[::Nf] / 3600 / 24
        xlabel = 't [days]'
    else:
        tp = t[::Nf]
        xlabel = 't'
        print(f"unknown tunits {tunits}, using whatever is in array")
    m = ax.pcolormesh(tp, f[:Nf], z.reshape((Nt,Nf)).T, cmap=cmap, norm=norm, shading='auto',snap=True, rasterized=True)
    if logf:
        ax.set_yscale('log')
    if title is not None:
        plt.title(title)
    fig.colorbar(m,ax=ax)
    plt.xlabel(xlabel)
    plt.ylabel('f')
    if savefname is not None:
        plt.savefig(savefname)
    plt.show()

In [ ]:
rundir = rundirs[1]
datadir = os.path.join(rundir,"data")
datadir_fullres = "wdmruns-coarse4/noise-conf-ln-1-detection-long/data/"
print(os.listdir(datadir))
full_noise = np.loadtxt(os.path.join(datadir, 'full_noise_model.dat'))
full_noise_fullres = np.loadtxt(os.path.join(datadir_fullres, 'full_noise_model.dat'))
print(full_noise.shape)
print(full_noise_fullres.shape)
Nt, Nf = get_Nt_Nf(full_noise_fullres)
#plt.imshow(full_noise_fullres[:,2].reshape(Nt,Nf).T, aspect='auto', origin='lower')
wavelet_specgram(full_noise_fullres[:,0], full_noise_fullres[:,1],  full_noise_fullres[:,2], Nt=Nt, cmap='viridis', logf=False, title="TDI X Dynamic PSD", tunits='days', savefname="full_dynamic_psd.png")

In [ ]:
Nt, Nf = get_Nt_Nf(full_noise)
wavelet_specgram(full_noise[:,0], full_noise[:,1],  full_noise[:,2], Nt=Nt)

In [ ]:
Nf

# compare coarse runs -- lognormal

In [ ]:
rundirs  = [
            #'wdmruns-coarse2/stat-noise-conf-ln-detection/',
            #'wdmruns-coarse3/stat-noise-conf-ln-detection/',
            #'wdmruns-coarse3/noise-conf-ln-169-detection/',
            #'wdmruns-coarse3/noise-conf-ln-ws169-detection/',
            #'wdmruns-coarse4/noise-conf-ln-169-detection/',
            #'wdmruns-coarse6/stat-noise-conf-ln-detection/',
            #'wdmruns-coarse3/noise-conf-ln-ws169-avg-detection/',
            'wdmruns-coarse6/noise-conf-ln-169-detection/',
           ]
chaindirs = [os.path.join(rundir, 'chains') for rundir in rundirs]
labels = [
         #'stationary',
         #'Q=169',
         'non-stationary',
         #'stationary (WS)',
         'Q=1',
         ]
colors = [
          'black',
          'red',
          'blue',
         ]
rowlims = [
    None,
    None,
    8700,
]
burn = 0

In [ ]:
rundirs  = [
            'wdmruns-coarse5/noise-conf-ln-4',
            'wdmruns-coarse5/stat-noise-conf-ln-4/',
            'wdmruns-coarse5/noise-conf-ln-35',
            'wdmruns-coarse5/stat-noise-conf-ln-35/',
            'wdmruns-coarse5/noise-conf-ln-3/',
            'wdmruns-coarse5/stat-noise-conf-ln-3/',
            #'wdmruns-coarse5/noise-conf-ln-25/',
           ]
chaindirs = [os.path.join(rundir, 'chains') for rundir in rundirs]
labels = [
         'A=-4',
         'A=-4 (stationary)',
         'A=-3.5',
         'A=-3.5 (stationary)',
        'A=-3',
        'A=-3 (stationary)'
         ]
colors = [
          'black',
          'red',
          'blue',
          'yellow',
          'green',
          'orange',
         ]
rowlims = [
    None,
    None,
    None,
    None,
    None,
    None,
]
burn = 0

In [ ]:
inst_labels = [r'$\log S_{\mathrm{acc},12}$',
r'$\log S_{\mathrm{acc},21}$',
r'$\log S_{\mathrm{acc},13}$',
r'$\log S_{\mathrm{acc},31}$',
r'$\log S_{\mathrm{acc},23}$', 
r'$\log S_{\mathrm{acc},32}$', 
r'$\log S_{\mathrm{oms},12}$', 
r'$\log S_{\mathrm{oms},21}$', 
r'$\log S_{\mathrm{oms},13}$', 
r'$\log S_{\mathrm{oms},31}$', 
r'$\log S_{\mathrm{oms},23}$', 
r'$\log S_{\mathrm{oms},32}$']

In [ ]:
galx_labels = [r'$\log A$', r'$f_1$', r'$\alpha$', r'$f_k$', r'$f_2$']
sgwb_labels = [r'$\log A_{\mathcal{R}}$', r'$\log f_\star$', r'$\log \Delta$']

In [ ]:
inst = np.loadtxt(os.path.join(chaindirs[0], 'noise_chain.dat'),max_rows=rowlims[0])
inst_chain = np.array(
[
    inst[burn:,i+2]
    for i in range(12)
]).T
truths = inst[0,2:]
old_corner = corner(inst_chain, labels = inst_labels, truths = truths, truth_color='purple', label=labels[0], color=colors[0], hist_kwargs={'density':True})

for (chdir, label, color, rows) in zip(chaindirs[1:], labels[1:], colors[1:], rowlims[1:]):
    inst = np.loadtxt(os.path.join(chdir, 'noise_chain.dat'), max_rows=rows)
    inst_chain = np.array(
    [
        inst[burn:,i+2]
        for i in range(12)
    ]).T
    truths = inst[0,2:]
    corner(inst_chain, fig= old_corner, labels = inst_labels, color=color, label=label, hist_kwargs={'density':True})
    

In [ ]:
galx = np.loadtxt(os.path.join(chaindirs[0], 'foreground_chain.dat'), max_rows=rowlims[0])
galx_chain = np.array(
[
    galx[burn:,i+2]
    for i in range(5)
]).T
truths = galx[0,2:]
old_corner = corner(galx_chain, labels = galx_labels, truths = truths, truth_color='purple', label=labels[0], color=colors[0], hist_kwargs={'density':True})

for (chdir, label, color, rows) in zip(chaindirs[1:], labels[1:], colors[1:], rowlims[1:]):
    galx = np.loadtxt(os.path.join(chdir, 'foreground_chain.dat'), max_rows=rows)
    galx_chain = np.array(
    [
        galx[burn:,i+2]
        for i in range(5)
    ]).T
    corner(galx_chain, fig= old_corner, labels = galx_labels, color=color, label=label, hist_kwargs={'density':True})
    

In [ ]:
sgwb = np.loadtxt(os.path.join(chaindirs[0], 'sgwb_chain.dat'), max_rows=rowlims[0])
sgwb_chain = np.array(
[
    sgwb[burn:,i+2]
    for i in range(sgwb.shape[1]-2)
]).T
truths = sgwb[0,2:]

bins = np.linspace(-5,-2,100)

plt.hist(sgwb_chain, density=True, alpha=0.5, label=labels[0], color=colors[0], bins=bins)

for (chdir, label, color, rows) in zip(chaindirs[1:2], labels[1:2], colors[1:2], rowlims[1:2]):
    sgwb = np.loadtxt(os.path.join(chdir, 'sgwb_chain.dat'), max_rows=rows)
    sgwb_chain = np.array(
    [
        sgwb[burn:,i+2]
        for i in range(sgwb.shape[1]-2)
    ]).T
    plt.hist(sgwb_chain, density=True, alpha=0.5, label=label, color=color, bins=bins)

plt.legend()

In [ ]:
sgwb = np.loadtxt(os.path.join(chaindirs[0], 'sgwb_chain.dat'), max_rows=rowlims[0])
sgwb_chain = np.array(
[
    sgwb[burn:,i+2]
    for i in range(sgwb.shape[1]-2)
]).T
truths = sgwb[0,2:]
old_corner = corner(sgwb_chain, labels = sgwb_labels, truths = truths, truth_color='green', label=labels[0], color=colors[0], hist_kwargs={'density':True})

for (chdir, label, color, rows) in zip(chaindirs[1:], labels[1:], colors[1:], rowlims[1:]):
    sgwb = np.loadtxt(os.path.join(chdir, 'sgwb_chain.dat'), max_rows=rows)
    sgwb_chain = np.array(
    [
        sgwb[burn:,i+2]
        for i in range(sgwb.shape[1]-2)
    ]).T
    corner(sgwb_chain, fig= old_corner, labels = sgwb_labels, color=color, label=label, hist_kwargs={'density':True})

    

In [ ]:
c = chainconsumer.ChainConsumer()
for chaindir,label in zip(chaindirs, labels):
    sgwb = np.loadtxt(os.path.join(chaindir, 'sgwb_chain.dat'))
    
    sgwb_chain = np.array(
    [
        sgwb[burn:,i+2]
        for i in range(3)
    ]).T
    sgwb_df = pd.DataFrame(sgwb_chain, columns = sgwb_labels)
    truths = sgwb[0,2:]
    c.add_chain(Chain(samples=sgwb_df, name=label))
c.add_truth(Truth(location = {sgwb_labels[i] : truths[i] for i in range(3)}))
c.plotter.set_config(
    PlotConfig(
        contour_label_font_size=16,
        dpi=100,
        legend_kwargs={'fontsize':20},#, 'loc': 'lower right'},
        plot_hists=False,
    )
)
c.plotter.plot()
plt.savefig("lognormal_contour.png")

# compare coarse runs -- phase transition

In [ ]:
rundirs  = [#'wdmruns-coarse/stat-noise-conf-pt/',
            #'wdmruns-coarse/noise-conf-pl-13/',
            #'wdmruns-coarse/noise-conf-pt-169/',
            #'wdmruns-coarse3/stat-noise-conf-pt/',
            #'wdmruns-coarse3/noise-conf-pt-169/',
            'wdmruns-coarse6/stat-noise-conf-pt/',
            'wdmruns-coarse6/noise-conf-pt-169/',
           ]
chaindirs = [os.path.join(rundir, 'chains') for rundir in rundirs]
labels = ['']*4
    #'stationary',
         #'Ncoarse=312',
         #'Ncoarse=24',
         #]
colors = ['black',
          'red',
          'blue',
          'yellow',
         ]
rowlims = [
    None,
    None,
    None,
    None
]

In [ ]:
inst_labels = [r'$\log S_{\mathrm{acc},12}$',
r'$\log S_{\mathrm{acc},21}$',
r'$\log S_{\mathrm{acc},13}$',
r'$\log S_{\mathrm{acc},31}$',
r'$\log S_{\mathrm{acc},23}$', 
r'$\log S_{\mathrm{acc},32}$', 
r'$\log S_{\mathrm{oms},12}$', 
r'$\log S_{\mathrm{oms},21}$', 
r'$\log S_{\mathrm{oms},13}$', 
r'$\log S_{\mathrm{oms},31}$', 
r'$\log S_{\mathrm{oms},23}$', 
r'$\log S_{\mathrm{oms},32}$']

In [ ]:
galx_labels = [r'$\log A$', r'$f_1$', r'$\alpha$', r'$f_k$', r'$f_2$']
# rb, b, log10 Ap, log10 fp [Hz]

sgwb_labels = [r'$r_b$', r'$b$', r'$\log A_p$', r'$\log f_p$']
burn = 0

In [ ]:
inst = np.loadtxt(os.path.join(chaindirs[0], 'noise_chain.dat'), max_rows=rowlims[0])
inst_chain = np.array(
[
    inst[burn:,i+2]
    for i in range(12)
]).T
truths = inst[0,2:]
old_corner = corner(inst_chain, labels = inst_labels, truths = truths, truth_color='red', label=labels[0], color=colors[0], hist_kwargs={'density':True})

for (chdir, label, color, rows) in zip(chaindirs[1:], labels[1:], colors[1:], rowlims[1:]):
    inst = np.loadtxt(os.path.join(chdir, 'noise_chain.dat'), max_rows=rows)
    inst_chain = np.array(
    [
        inst[burn:,i+2]
        for i in range(12)
    ]).T
    truths = inst[0,2:]
    corner(inst_chain, fig= old_corner, labels = inst_labels, color=color, label=label, hist_kwargs={'density':True})
    

In [ ]:
galx = np.loadtxt(os.path.join(chaindirs[0], 'foreground_chain.dat'), max_rows=rowlims[0])
galx_chain = np.array(
[
    galx[burn:,i+2]
    for i in range(5)
]).T
truths = galx[0,2:]
old_corner = corner(galx_chain, labels = galx_labels, truths = truths, truth_color='red', label=labels[0], color=colors[0], hist_kwargs={'density':True})

for (chdir, label, color, rows) in zip(chaindirs[1:], labels[1:], colors[1:], rowlims[1:]):
    galx = np.loadtxt(os.path.join(chdir, 'foreground_chain.dat'), max_rows=rows)
    galx_chain = np.array(
    [
        galx[burn:,i+2]
        for i in range(5)
    ]).T
    corner(galx_chain, fig= old_corner, labels = galx_labels, color=color, label=label, hist_kwargs={'density':True})
    

In [ ]:
sgwb = np.loadtxt(os.path.join(chaindirs[0], 'sgwb_chain.dat'), max_rows=rowlims[0])
sgwb_chain = np.array(
[
    sgwb[burn:,i+2]
    for i in range(4)
]).T
truths = sgwb[0,2:]
old_corner = corner(sgwb_chain, labels = sgwb_labels, truths = truths, truth_color='red', label=labels[0], color=colors[0], hist_kwargs={'density':True})

for (chdir, label, color, rows) in zip(chaindirs[1:], labels[1:], colors[1:], rowlims[1:]):
    sgwb = np.loadtxt(os.path.join(chdir, 'sgwb_chain.dat'), max_rows=rows)
    sgwb_chain = np.array(
    [
        sgwb[burn:,i+2]
        for i in range(4)
    ]).T
    corner(sgwb_chain, fig= old_corner, labels = sgwb_labels, color=color, label=label, hist_kwargs={'density':True})

    

# varying noise model runs

In [ ]:
rundirs  = [
            #'wdmruns-drift/null/',
            #'wdmruns-drift/oms-only/',
            #'wdmruns-coarse6/noise-conf-pl-169/',
            'wdmruns-drift/with-pl-prior-1/'
           ]
chaindirs = [os.path.join(rundir, 'chains') for rundir in rundirs]
labels = [
         #'stationary',
         #'Q=169',
         'null',
         #'oms only',
         'all over'
         #'stationary (WS)',
         #'Q=1',
         ]
colors = [
          'black',
          #'red',
          'blue',
         ]
rowlims = [
    None,
    None,
    None,
]

inst_truths = [
    #[5.76e-30]*6 + [1.28e-22]*6 + [0.0]*12,
   # [5.76e-30]*6 + [1.28e-22]*6 + [0.0]*6 + [0.2]*6,
    [5.76e-30]*6 + [1.28e-22]*6 + [0.7,0.4,-0.2,-0.2,0.1,0.1,0.7,0.3,-0.1,-0.7,0.0,0.7],
]

# initialize time dependent foreground parameter elves
af1 = -2.235e-1;
bf1 = -2.7040844;
afk = -3.60976122e-1;
bfk = -2.37822436;

Tobs = 2 # years
f1  =  pow(10., af1*np.log10(Tobs) + bf1)
fk  =  pow(10., afk*np.log10(Tobs) + bfk)


galx_truths = [
    [np.log(1.2826e-44), np.log(f1), 1.629667, np.log(fk), np.log(4.810781e-4)]

]

sgwb_truths = [
    [-20.0,2/3.]
]

burn = 1000

In [ ]:
inst_labels = [
    r'$\log S_{\mathrm{acc},12}$',
    r'$\log S_{\mathrm{acc},21}$',
    r'$\log S_{\mathrm{acc},13}$',
    r'$\log S_{\mathrm{acc},31}$',
    r'$\log S_{\mathrm{acc},23}$', 
    r'$\log S_{\mathrm{acc},32}$', 
    r'$\log S_{\mathrm{oms},12}$', 
    r'$\log S_{\mathrm{oms},21}$', 
    r'$\log S_{\mathrm{oms},13}$', 
    r'$\log S_{\mathrm{oms},31}$', 
    r'$\log S_{\mathrm{oms},23}$', 
    r'$\log S_{\mathrm{oms},32}$',
    r'$\alpha_{\mathrm{acc},12}$',
    r'$\alpha_{\mathrm{acc},21}$',
    r'$\alpha_{\mathrm{acc},13}$',
    r'$\alpha_{\mathrm{acc},31}$',
    r'$\alpha_{\mathrm{acc},23}$', 
    r'$\alpha_{\mathrm{acc},32}$', 
    r'$\alpha_{\mathrm{oms},12}$', 
    r'$\alpha_{\mathrm{oms},21}$', 
    r'$\alpha_{\mathrm{oms},13}$', 
    r'$\alpha_{\mathrm{oms},31}$', 
    r'$\alpha_{\mathrm{oms},23}$', 
    r'$\alpha_{\mathrm{oms},32}$',
              ]
galx_labels = [r'$\log A$', r'$\log f_1$', r'$\alpha$', r'$\log f_k$', r'$\log f_2$']
sgwb_labels = [r'$\log A_p$', r'$\alpha_p$']

In [ ]:
os.listdir(chaindirs[0])

In [ ]:
# acc corner

inst = np.loadtxt(os.path.join(chaindirs[0], 'noise_chain.dat'), max_rows=rowlims[0])
ndim = 6
estart = 0
eend = estart+ndim
inst_chain = np.array(
[
    inst[burn:,i+2]
    for i in range(estart,eend)
]).T
truths = inj_truths[0][estart:eend]
old_corner = corner(inst_chain, labels = inst_labels[estart:eend], truths = truths, truth_color='red', label=labels[0], color=colors[0], hist_kwargs={'density':True})

for (chdir, label, color, rows) in zip(chaindirs[1:], labels[1:], colors[1:], rowlims[1:]):
    inst = np.loadtxt(os.path.join(chdir, 'noise_chain.dat'), max_rows=rows)
    inst_chain = np.array(
    [
        inst[burn:,i+2]
        for i in range(estart,eend)
    ]).T
    truths = inst[0,2:]
    corner(inst_chain, fig= old_corner, labels = inst_labels[estart:eend], color=color, label=label, hist_kwargs={'density':True})

In [ ]:
# oms corner

inst = np.loadtxt(os.path.join(chaindirs[0], 'noise_chain.dat'), max_rows=rowlims[0])
ndim = 6
estart = 6
eend = estart+ndim
inst_chain = np.array(
[
    inst[burn:,i+2]
    for i in range(estart,eend)
]).T
truths = inj_truths[0][estart:eend]
old_corner = corner(inst_chain, labels = inst_labels[estart:eend], truths = truths, truth_color='red', label=labels[0], color=colors[0], hist_kwargs={'density':True})

for (chdir, label, color, rows) in zip(chaindirs[1:], labels[1:], colors[1:], rowlims[1:]):
    inst = np.loadtxt(os.path.join(chdir, 'noise_chain.dat'), max_rows=rows)
    inst_chain = np.array(
    [
        inst[burn:,i+2]
        for i in range(estart,eend)
    ]).T
    truths = inst[0,2:]
    corner(inst_chain, fig= old_corner, labels = inst_labels[estart:eend], color=color, label=label, hist_kwargs={'density':True})

In [ ]:
# alpha acc corner

inst = np.loadtxt(os.path.join(chaindirs[0], 'noise_chain.dat'), max_rows=rowlims[0])
ndim = 6
estart = 12
eend = estart+ndim
inst_chain = np.array(
[
    inst[burn:,i+2]
    for i in range(estart,eend)
]).T
truths = inst_truths[0][estart:eend]
print(truths)
old_corner = corner(inst_chain, labels = inst_labels[estart:eend], truths = truths, truth_color='red', label=labels[0], color=colors[0], hist_kwargs={'density':True})

for (chdir, label, color, rows, truths) in zip(chaindirs[1:], labels[1:], colors[1:], rowlims[1:], inst_truths[1:]):
    inst = np.loadtxt(os.path.join(chdir, 'noise_chain.dat'), max_rows=rows)
    inst_chain = np.array(
    [
        inst[burn:,i+2]
        for i in range(estart,eend)
    ]).T
    truths = truths[estart:eend]
    corner(inst_chain, fig= old_corner, labels = inst_labels[estart:eend], color=color, label=label, hist_kwargs={'density':True})

In [ ]:
# alpha oms corner

inst = np.loadtxt(os.path.join(chaindirs[0], 'noise_chain.dat'), max_rows=rowlims[0])
ndim = 6
estart = 18
eend = estart+ndim
inst_chain = np.array(
[
    inst[burn:,i+2]
    for i in range(estart,eend)
]).T
truths = inst_truths[0][estart:eend]
truths = [0.5,0.5, -0.4,-0.4,0.35,0.35]
print(truths)
old_corner = corner(inst_chain, labels = inst_labels[estart:eend], truths = truths, truth_color='red', label=labels[0], color=colors[0], hist_kwargs={'density':True})

for (chdir, label, color, rows, truths) in zip(chaindirs[1:], labels[1:], colors[1:], rowlims[1:], inst_truths[1:]):
    inst = np.loadtxt(os.path.join(chdir, 'noise_chain.dat'), max_rows=rows)
    inst_chain = np.array(
    [
        inst[burn:,i+2]
        for i in range(estart,eend)
    ]).T
    truths = truths[estart:eend]
    corner(inst_chain, fig= old_corner, labels = inst_labels[estart:eend], truths=truths, color=color, label=label, hist_kwargs={'density':True})

In [ ]:
galx = np.loadtxt(os.path.join(chaindirs[0], 'foreground_chain.dat'), max_rows=rowlims[0])
galx_chain = np.array(
[
    galx[burn:,i+2]
    for i in range(5)
]).T
truths = galx_truths[0]
old_corner = corner(galx_chain, labels = galx_labels, truths = truths, truth_color='red', label=labels[0], color=colors[0], hist_kwargs={'density':True})

for (chdir, label, color, rows) in zip(chaindirs[1:], labels[1:], colors[1:], rowlims[1:]):
    galx = np.loadtxt(os.path.join(chdir, 'foreground_chain.dat'), max_rows=rows)
    galx_chain = np.array(
    [
        galx[burn:,i+2]
        for i in range(5)
    ]).T
    corner(galx_chain, fig= old_corner, labels = galx_labels, color=color, label=label, hist_kwargs={'density':True})

In [ ]:
sgwb = np.loadtxt(os.path.join(chaindirs[0], 'sgwb_chain.dat'), max_rows=rowlims[0])
sgwb_chain = np.array(
[
    sgwb[burn:,i+2]
    for i in range(2)
]).T
truths = sgwb_truths[0]
old_corner = corner(sgwb_chain, labels = sgwb_labels, truths = truths, truth_color='red', label=labels[0], color=colors[0], hist_kwargs={'density':True})

for (chdir, label, color, rows) in zip(chaindirs[1:], labels[1:], colors[1:], rowlims[1:]):
    sgwb = np.loadtxt(os.path.join(chdir, 'sgwb_chain.dat'), max_rows=rows)
    sgwb_chain = np.array(
    [
        sgwb[burn:,i+2]
        for i in range(2)
    ]).T
    corner(sgwb_chain, fig= old_corner, labels = sgwb_labels, color=color, label=label, hist_kwargs={'density':True})

In [ ]:
import matplotlib_inline.backend_inline
matplotlib_inline.backend_inline.set_matplotlib_formats('svg')
from matplotlib import colors
%matplotlib inline

def get_Nt_Nf(datafile_arr):
    # cols are t, f, ...
    Nf = np.unique(datafile_arr[:,1]).shape[0]
    Nt = np.unique(datafile_arr[:,0]).shape[0]
    assert Nt*Nf == datafile_arr.shape[0]
    return Nt,Nf

# TODO: separate real/imag FFT coeffs in each layer
def wavelet_specgram(t, f, pz, Nt = 512, clip = 0, lognorm=False, logf=False, cmap='bwr', cx_norm='abs', title=None, tunits='s', savefname=None):
    Nf = t.shape[0] // Nt
    if t.shape[0] != Nt*Nf or f.shape[0] != Nt*Nf:
        print("Nt or Nf is wrong")
    z = pz.copy()
    if lognorm:
        z = np.ma.masked_less_equal(z.astype(float), 0)
        lo, hi = np.nanpercentile(z.compressed(), [clip, 100-clip])
        norm = colors.AsinhNorm(vmin=lo, vmax = hi, clip=True)
    else:
        lo, hi = np.nanpercentile(z, [clip, 100-clip])
        norm = colors.Normalize(vmin=lo, vmax = hi, clip=True)
    #plt.imshow(z.reshape((Nf,Nt)),cmap=cmap,norm=norm,aspect=9/16*Nt/Nf,interpolation='none')
    fig, ax = plt.subplots()
    if tunits == 's':
        tp = t[::Nf]
        xlabel = 't [s]'
    elif tunits == 'days':
        tp = t[::Nf] / 3600 / 24
        xlabel = 't [days]'
    else:
        tp = t[::Nf]
        xlabel = 't'
        print(f"unknown tunits {tunits}, using whatever is in array")
    m = ax.pcolormesh(tp, f[:Nf], z.reshape((Nt,Nf)).T, cmap=cmap, norm=norm, shading='auto',snap=True, rasterized=True)
    if logf:
        ax.set_yscale('log')
    if title is not None:
        plt.title(title)
    fig.colorbar(m,ax=ax)
    plt.xlabel(xlabel)
    plt.ylabel('f')
    if savefname is not None:
        plt.savefig(savefname)
    plt.show()

rundir = rundirs[0]
datadir = os.path.join(rundir,"data")
#datadir_fullres = "wdmruns-coarse4/noise-conf-ln-1-detection-long/data/"
full_noise = np.loadtxt(os.path.join(datadir, 'full_noise_model.dat'))
full_noise2 = np.loadtxt(os.path.join('wdmruns-drift/with-pl-prior-1-cheat/' , 'data/full_noise_model.dat'))
#full_noise_fullres = np.loadtxt(os.path.join(datadir_fullres, 'full_noise_model.dat'))
print(full_noise.shape)
#print(full_noise_fullres.shape)
Nt, Nf = get_Nt_Nf(full_noise)
#plt.imshow(full_noise_fullres[:,2].reshape(Nt,Nf).T, aspect='auto', origin='lower')
#wavelet_specgram(full_noise_fullres[:,0], full_noise_fullres[:,1],  full_noise_fullres[:,2], Nt=Nt, cmap='viridis', logf=False, title="TDI X Dynamic PSD", tunits='days', savefname="full_dynamic_psd.png")
wavelet_specgram(full_noise[:,0], full_noise[:,1],  full_noise2[:,2], Nt=Nt, cmap='viridis', logf=False, title="TDI X Dynamic PSD", tunits='days', savefname="full_dynamic_psd.png")

In [ ]:
wavelet_specgram(full_noise[:,0], full_noise[:,1],  full_noise2[:,2] - full_noise[:,2], Nt=Nt, cmap='viridis', logf=False, title="TDI X Dynamic PSD", tunits='days')# savefname="full_dynamic_psd.png")